In [ ]:
# -*- coding: utf-8 -*-
"""
Incendis forestals a Catalunya (1998-2024) — Visualitzacions Finals Q1-Q4
Information Visualization | Projecte 1
Inspiració d'estil: Periodisme de dades (Roger Tugas)
Tots els fitxers generats acaben amb el sufix '_vg.html'.
"""

import copy
import json
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# %% ===================================================================
# 0. CÀRREGA I NETEJA DE DADES
# =======================================================================
df = pd.read_csv('incendis_final.csv')
df.head

num_cols = [
    'HAARBRADES', 'HANOARBRAD', 'SuperficieTotalForestal',
    'HANOFOREST', 'SuperficieAgricola',
    'OtrasSuperficiesNoforestales', 'SuperficieTotalNoForestal'
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['SuperficieTotal'] = (
    df['SuperficieTotalForestal'].fillna(0)
    + df['SuperficieTotalNoForestal'].fillna(0)
)


def any_des_de_data(serie):
    dates = pd.to_datetime(serie, dayfirst=True, errors='coerce', format='mixed')
    return dates.dt.year


any_1 = any_des_de_data(df['Data incendi'])
any_2 = any_des_de_data(df['Data Incendi'])
any_campania = df['Campania'].astype(str).str.extract(r'(\d{4})\s*$')[0].astype(float)
df['Any'] = any_1.combine_first(any_2).combine_first(any_campania)

df = df.dropna(subset=['Nom comarca', 'Any']).copy()
df['Any'] = df['Any'].astype(int)
df = df[(df['Any'] >= 1998) & (df['Any'] <= 2024)]

# Unifiquem causes buides o "Desconeguda" amb la categoria "Desconocida" real del CSV
_causes_valides = df['Causa'].dropna().astype(str).str.strip()
_candidata_desconeguda = next(
    (c for c in _causes_valides.unique() if 'esconoc' in c.lower()),
    next((c for c in _causes_valides.unique() if 'esconeg' in c.lower()), 'Desconocida')
)
df['Causa'] = df['Causa'].fillna(_candidata_desconeguda).astype(str).str.strip()
df['Causa'] = df['Causa'].replace({
    'nan': _candidata_desconeguda,
    '': _candidata_desconeguda,
    'Desconeguda': _candidata_desconeguda,
    'desconeguda': _candidata_desconeguda
})

for c in ['Nom comarca', 'Nom vegueria']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

correccions_comarca = {
    'Aran': "Val d'Aran",
    'Montisà': 'Montsià',
    "Val d'Aran": "Val d'Aran"
}
df['Nom comarca'] = df['Nom comarca'].replace(correccions_comarca)

top_causes = df['Causa'].value_counts().nlargest(6).index
df['CausaAgrupada'] = df['Causa'].where(df['Causa'].isin(top_causes), 'Altres')
causa_domain = sorted(df['CausaAgrupada'].unique())
causa_scale = alt.Scale(domain=causa_domain, scheme='dark2')

# --- Classes de mida de l'incendi (Q4) ---
classes_mida = ['<1 ha', '1-10 ha', '10-100 ha', '100-500 ha', '>500 ha']
df['ClasseMida'] = pd.cut(
    df['SuperficieTotal'],
    bins=[-0.01, 1, 10, 100, 500, 1e9],
    labels=classes_mida
)
colors_mida_vius = ['#fdae61', '#f46d43', '#d73027', '#7b3294', '#1a1a1a']
mida_scale = alt.Scale(domain=classes_mida, range=colors_mida_vius)

# --- Períodes per a Q3 (9 triennis, 1998-2024 -> graella 3x3) ---
bins_q3 = [1998, 2001, 2004, 2007, 2010, 2013, 2016, 2019, 2022, 2025]
labels_q3 = [
    '1998-2000', '2001-2003', '2004-2006',
    '2007-2009', '2010-2012', '2013-2015',
    '2016-2018', '2019-2021', '2022-2024'
]
df['PeriodeQ3'] = pd.cut(df['Any'], bins=bins_q3, labels=labels_q3, right=False)

# --- Períodes per a Q4 (6 períodes) ---
bins_q4 = [1998, 2003, 2008, 2013, 2018, 2022, 2025]
labels_q4 = ['1998-2002', '2003-2007', '2008-2012', '2013-2017', '2018-2021', '2022-2024']
df['PeriodeQ4'] = pd.cut(df['Any'], bins=bins_q4, labels=labels_q4, right=False)

# --- GeoJSON i càlcul d'aspect ratio real ---
geojson_path = (
    'divisions-administratives-v2r2-20260120/'
    'divisions-administratives-v2r2-comarques-500000-20260120.json'
)
with open(geojson_path, encoding='utf-8') as f:
    geo_dict = json.load(f)
comarques_geo = alt.Data(values=geo_dict['features'])
noms_geojson = {feat['properties']['NOMCOMAR'] for feat in geo_dict['features']}


def _calcular_bbox(geo_d):
    xs, ys = [], []

    def recorre(coords):
        if isinstance(coords[0], (int, float)):
            xs.append(coords[0])
            ys.append(coords[1])
        else:
            for c in coords:
                recorre(c)

    for feat in geo_d['features']:
        recorre(feat['geometry']['coordinates'])
    return min(xs), max(xs), min(ys), max(ys)


_minx, _maxx, _miny, _maxy = _calcular_bbox(geo_dict)
ASPECT_RATIO = (_maxx - _minx) / (_maxy - _miny)


def mida_mapa(amplada):
    return amplada, round(amplada / ASPECT_RATIO)


PROJECCIO = 'identity'
REFLECTEIX_Y = True


def titol_rt(titol, subtitol):
    return alt.TitleParams(
        text=titol,
        subtitle=subtitol,
        anchor='start',
        fontSize=20,
        fontWeight='bold',
        subtitleFontSize=13,
        subtitleColor='#555555',
        offset=12
    )


# %% ===================================================================
# Q1 · v1 (2 Mapes: Hectàries i Nombre d'incendis) + v2 (1 Mapa amb distorsió)
# =======================================================================
df_comarques = df.groupby('Nom comarca', as_index=False).agg(
    Superficie_Total_Calcinada=('SuperficieTotal', 'sum'),
    Nombre_Incendis=('SuperficieTotal', 'count'),
    Mida_Mitjana=('SuperficieTotal', 'mean')
).round(2)

df_comarques_complet = pd.DataFrame({'Nom comarca': sorted(list(noms_geojson))}).merge(
    df_comarques, on='Nom comarca', how='left'
).fillna({'Superficie_Total_Calcinada': 0, 'Nombre_Incendis': 0, 'Mida_Mitjana': 0})

w1, h1 = mida_mapa(540)

mapa_q1_inc = alt.Chart(comarques_geo).mark_geoshape(
    stroke='white', strokeWidth=0.9
).transform_lookup(
    lookup='properties.NOMCOMAR',
    from_=alt.LookupData(
        df_comarques_complet, 'Nom comarca',
        ['Superficie_Total_Calcinada', 'Nombre_Incendis', 'Mida_Mitjana']
    )
).encode(
    color=alt.Color(
        'Nombre_Incendis:Q',
        scale=alt.Scale(scheme='oranges'),
        title="Nombre d'incendis",
        legend=alt.Legend(orient='bottom', gradientLength=260, titleFontSize=13, labelFontSize=12)
    ),
    tooltip=[
        alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
        alt.Tooltip('Nombre_Incendis:Q', title="Nombre d'incendis", format=','),
        alt.Tooltip('Superficie_Total_Calcinada:Q', title='Superfície calcinada (ha)', format=',.1f')
    ]
).project(type=PROJECCIO, reflectY=REFLECTEIX_Y).properties(
    width=w1, height=h1,
    title=alt.TitleParams("Freqüència: Nombre total d'incendis per comarca", fontSize=15, fontWeight='bold')
)

mapa_q1_ha = alt.Chart(comarques_geo).mark_geoshape(
    stroke='white', strokeWidth=0.9
).transform_lookup(
    lookup='properties.NOMCOMAR',
    from_=alt.LookupData(
        df_comarques_complet, 'Nom comarca',
        ['Superficie_Total_Calcinada', 'Nombre_Incendis', 'Mida_Mitjana']
    )
).encode(
    color=alt.Color(
        'Superficie_Total_Calcinada:Q',
        scale=alt.Scale(scheme='reds', type='symlog', constant=50),
        title='Hectàrees calcinades (ha)',
        legend=alt.Legend(orient='bottom', gradientLength=260, titleFontSize=13, labelFontSize=12)
    ),
    tooltip=[
        alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
        alt.Tooltip('Superficie_Total_Calcinada:Q', title='Superfície calcinada (ha)', format=',.1f'),
        alt.Tooltip('Nombre_Incendis:Q', title="Nombre d'incendis", format=',')
    ]
).project(type=PROJECCIO, reflectY=REFLECTEIX_Y).properties(
    width=w1, height=h1,
    title=alt.TitleParams("Impacte: Hectàrees totals calcinades per comarca", fontSize=15, fontWeight='bold')
)

q1_v1 = alt.hconcat(mapa_q1_inc, mapa_q1_ha, spacing=24).resolve_scale(color='independent').properties(
    title=titol_rt(
        "Q1 (v1) · Freqüència vs. Impacte: on s'encenen més focs no sempre és on es crema més bosc",
        "Comparativa per comarca del nombre total d'incendis (esquerra) i les hectàrees calcinades (dreta) a Catalunya (1998-2024)"
    )
).configure_view(stroke=None)
q1_v1.save('Q1_v1_dos_mapes_vg.html')


# --- Q1 v2: 1 sol mapa amb DISTORSIÓ (Cartograma no-contigu segons Nombre d'Incendis + Color = Hectàrees) ---
def crear_cartograma_distorsionat(geo_d, df_stats):
    """
    Deforma/escala el polígon de cada comarca respecte al seu centroide
    en funció del nombre d'incendis (Cartograma d'escala de polígons),
    mantenint la silueta de Catalunya reconeixible.
    """
    geo_copy = copy.deepcopy(geo_d)
    max_inc = max(df_stats['Nombre_Incendis'].max(), 1)
    inc_map = df_stats.set_index('Nom comarca')['Nombre_Incendis'].to_dict()

    for feat in geo_copy['features']:
        nom = feat['properties']['NOMCOMAR']
        n_inc = inc_map.get(nom, 0)
        # Factor d'escala lineal amb arrel quadrada (perquè l'àrea sigui proporcional al nombre d'incendis)
        factor = 0.30 + 0.85 * np.sqrt(n_inc / max_inc)

        pts_x, pts_y = [], []

        def recolhir(coords):
            if isinstance(coords[0], (int, float)):
                pts_x.append(coords[0])
                pts_y.append(coords[1])
            else:
                for c in coords:
                    recolhir(c)

        recolhir(feat['geometry']['coordinates'])
        cx, cy = np.mean(pts_x), np.mean(pts_y)

        def escalar(coords):
            if isinstance(coords[0], (int, float)):
                return [cx + (coords[0] - cx) * factor, cy + (coords[1] - cy) * factor]
            return [escalar(c) for c in coords]

        feat['geometry']['coordinates'] = escalar(feat['geometry']['coordinates'])
    return geo_copy


geo_distorsionat = crear_cartograma_distorsionat(geo_dict, df_comarques_complet)
comarques_cartograma = alt.Data(values=geo_distorsionat['features'])
w1_carto, h1_carto = mida_mapa(720)

# Silueta de fons suau (mida real) + Comarques distorsionades segons nombre d'incendis i acolorides per hectàrees
fons_silueta = alt.Chart(comarques_geo).mark_geoshape(
    fill='#f2f2f2', stroke='#d0d0d0', strokeWidth=0.6
).project(type=PROJECCIO, reflectY=REFLECTEIX_Y)

capa_distorsio = alt.Chart(comarques_cartograma).mark_geoshape(
    stroke='#222222', strokeWidth=0.9, fillOpacity=0.92
).transform_lookup(
    lookup='properties.NOMCOMAR',
    from_=alt.LookupData(
        df_comarques_complet, 'Nom comarca',
        ['Superficie_Total_Calcinada', 'Nombre_Incendis', 'Mida_Mitjana']
    )
).encode(
    color=alt.Color(
        'Superficie_Total_Calcinada:Q',
        scale=alt.Scale(scheme='reds', type='symlog', constant=50),
        title='Superfície cremada (ha)',
        legend=alt.Legend(orient='right', gradientLength=260, titleFontSize=13, labelFontSize=12)
    ),
    tooltip=[
        alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
        alt.Tooltip('Nombre_Incendis:Q', title="Nombre d'incendis (mida polígon)", format=','),
        alt.Tooltip('Superficie_Total_Calcinada:Q', title='Hectàrees cremades (color)', format=',.1f'),
        alt.Tooltip('Mida_Mitjana:Q', title='Mida mitjana per foc (ha)', format=',.2f')
    ]
).project(type=PROJECCIO, reflectY=REFLECTEIX_Y)

q1_v2 = (fons_silueta + capa_distorsio).properties(
    width=w1_carto, height=h1_carto,
    title=titol_rt(
        "Q1 (v2) · Cartograma de distorsió: la mida de cada comarca reflecteix el nombre d'incendis i el color les hectàrees",
        "Les comarques amb més incendis s'expandeixen i les que en tenen pocs s'encongeixen sobre la silueta real (1998-2024)"
    )
).configure_view(stroke=None)
q1_v2.save('Q1_v2_distorsio_cartograma_vg.html')


# %% ===================================================================
# Q2 · Triennis (q2v1b) + Quadradets en 3 files etiquetades (Petits/Mitjans=20 focs, Grans=1 foc)
# =======================================================================
df_q2 = df[df['Any'] <= 2022].copy()
bins_trienni = [1998, 2001, 2004, 2007, 2010, 2013, 2016, 2019, 2023]
labels_trienni = [
    '1998-2000', '2001-2003', '2004-2006', '2007-2009',
    '2010-2012', '2013-2015', '2016-2018', '2019-2022'
]
df_q2['Trienni'] = pd.cut(df_q2['Any'], bins=bins_trienni, labels=labels_trienni, right=False)

AMPLADA_BARRA_Q2 = 540
ALCADA_BARRA_Q2 = 66
AMPLADA_LABEL_Q2 = 90
AMPLADA_BOLA_TOTAL_Q2 = 85
AMPLADA_REF_TRAM_Q2 = 105
AMPLADA_PICTO_Q2 = 360

tram_q2_domain = ['Petits (<1 ha)', 'Mitjans (1-500 ha)', 'Grans (>=500 ha)']
tram_q2_colors = alt.Scale(
    domain=tram_q2_domain,
    range=['#fdae61', '#f46d43', '#a50026']
)


def quadradets_periode_q2(subdf, periode):
    files = []
    # Fila 0 (a dalt): Grans (>= 500 ha) -> 1 quadradet = 1 incendi real
    grans = subdf[subdf['SuperficieTotal'] >= 500].sort_values('SuperficieTotal', ascending=False)
    for i, (_, r) in enumerate(grans.iterrows()):
        mun = r.get('Nom municipi', r.get('Nom comarca', ''))
        if pd.isna(mun) or str(mun).strip() == '':
            mun = r.get('Nom comarca', '')
        files.append({
            'Periode': periode,
            'Tram': 'Grans (>=500 ha)',
            'y_row': 0,
            'col': i,
            'mida_sq': 115,
            'Representa': '1 Gran Incendi Forestal',
            'Detall': f"{mun} ({int(r['Any'])}): {r['SuperficieTotal']:,.1f} ha"
        })

    # Fila 1 (al mig): Mitjans (1-500 ha) -> 1 columna/quadradet = 20 incendis
    n_mitjans = int(((subdf['SuperficieTotal'] >= 1) & (subdf['SuperficieTotal'] < 500)).sum())
    n_sq_mitjans = int(np.ceil(n_mitjans / 20.0)) if n_mitjans > 0 else 0
    for i in range(n_sq_mitjans):
        files.append({
            'Periode': periode,
            'Tram': 'Mitjans (1-500 ha)',
            'y_row': 1,
            'col': i,
            'mida_sq': 80,
            'Representa': '20 incendis mitjans (1-500 ha)',
            'Detall': f"Total al trienni: {n_mitjans:,} incendis mitjans"
        })

    # Fila 2 (a baix): Petits (< 1 ha) -> 1 columna/quadradet = 20 incendis
    n_petits = int((subdf['SuperficieTotal'] < 1).sum())
    n_sq_petits = int(np.ceil(n_petits / 20.0)) if n_petits > 0 else 0
    for i in range(n_sq_petits):
        files.append({
            'Periode': periode,
            'Tram': 'Petits (<1 ha)',
            'y_row': 2,
            'col': i,
            'mida_sq': 55,
            'Representa': '20 conats (< 1 ha)',
            'Detall': f"Total al trienni: {n_petits:,} conats"
        })

    return pd.DataFrame(files)


df_sq_q2 = pd.concat(
    [quadradets_periode_q2(df_q2[df_q2['Trienni'] == p], p) for p in labels_trienni],
    ignore_index=True
)
max_col_q2 = max(int(df_sq_q2['col'].max()) + 2, 45)

# Càlcul manual d'apilament de les barres de causes per trienni
df_causa_q2 = (
    df_q2.groupby(['Trienni', 'CausaAgrupada'], as_index=False, observed=True)
    .size().rename(columns={'size': 'n'})
)
parts_q2 = []
for idx_p, p in enumerate(labels_trienni):
    sub = df_causa_q2[df_causa_q2['Trienni'] == p].set_index('CausaAgrupada').reindex(causa_domain).reset_index()
    sub['n'] = sub['n'].fillna(0)
    tot = sub['n'].sum()
    sub['pct'] = sub['n'] / tot if tot else 0
    sub['x1'] = sub['pct'].cumsum()
    sub['x0'] = sub['x1'] - sub['pct']
    sub['x_mig'] = (sub['x0'] + sub['x1']) / 2
    sub['Trienni'] = p
    sub['label_txt'] = sub['pct'].apply(
        lambda v: (f"{round(v * 100)}%" if idx_p == 0 else f"{round(v * 100)}") if v >= 0.045 else ""
    )
    parts_q2.append(sub)
df_causa_q2 = pd.concat(parts_q2, ignore_index=True)

# Capçaleres de columna Q2
cap_q2_causes = alt.Chart(pd.DataFrame({'CausaAgrupada': causa_domain})).mark_square(size=180, opacity=0).encode(
    color=alt.Color(
        'CausaAgrupada:N', scale=causa_scale, title='Causa:',
        legend=alt.Legend(
            orient='top', direction='horizontal', columns=3,
            titleOrient='left', titleFontSize=13, labelFontSize=12,
            symbolType='circle', symbolSize=180, symbolOpacity=1
        )
    )
).properties(width=AMPLADA_BARRA_Q2, height=1)

cap_q2_any = alt.Chart(pd.DataFrame({'t': ['TRIENNI']})).mark_text(
    fontSize=13, fontWeight='bold', color='#222222'
).encode(text='t:N').properties(width=AMPLADA_LABEL_Q2, height=1)

cap_q2_tot = alt.Chart(pd.DataFrame({'t': ["Total focs"]})).mark_text(
    fontSize=12, fontWeight='bold', color='#222222'
).encode(text='t:N').properties(width=AMPLADA_BOLA_TOTAL_Q2, height=1)

cap_q2_ref = alt.Chart(pd.DataFrame({'t': ["Categoria mida"]})).mark_text(
    fontSize=12, fontWeight='bold', align='right', color='#222222'
).encode(text='t:N').properties(width=AMPLADA_REF_TRAM_Q2, height=1)

cap_q2_sq = alt.Chart(pd.DataFrame({'Tram': tram_q2_domain})).mark_square(size=180, opacity=0).encode(
    color=alt.Color(
        'Tram:N', scale=tram_q2_colors, title='Quadradets (Petits/Mitjans: 1 col = 20 focs | Grans: 1 = 1 GIF):',
        legend=alt.Legend(
            orient='top', direction='horizontal',
            titleOrient='top', titleFontSize=12, labelFontSize=12,
            symbolType='square', symbolSize=160, symbolOpacity=1
        )
    )
).properties(width=AMPLADA_PICTO_Q2, height=1)

files_q2 = [alt.hconcat(cap_q2_causes, cap_q2_any, cap_q2_tot, cap_q2_ref, cap_q2_sq, spacing=6)]

df_etiquetes_files_q2 = pd.DataFrame({
    'y_row': [0, 1, 2],
    'label_fila': ['Grans (>=500 ha)', 'Mitjans (1-500 ha)', 'Petits (<1 ha)']
})

for p in labels_trienni:
    sub_b = df_causa_q2[df_causa_q2['Trienni'] == p]
    c_barra = (
        alt.Chart(sub_b).mark_bar(height=ALCADA_BARRA_Q2).encode(
            x=alt.X('x0:Q', scale=alt.Scale(domain=[0, 1], nice=False), axis=None),
            x2='x1:Q',
            color=alt.Color('CausaAgrupada:N', scale=causa_scale, legend=None),
            tooltip=[
                alt.Tooltip('CausaAgrupada:N', title='Causa'),
                alt.Tooltip('n:Q', title="Nombre d'incendis", format=','),
                alt.Tooltip('pct:Q', title='Percentatge', format='.1%')
            ]
        ) + alt.Chart(sub_b[sub_b['pct'] >= 0.045]).mark_text(
            color='white', fontSize=13, fontWeight='bold', baseline='middle'
        ).encode(
            x=alt.X('x_mig:Q', scale=alt.Scale(domain=[0, 1], nice=False)),
            text='label_txt:N'
        )
    ).properties(width=AMPLADA_BARRA_Q2, height=ALCADA_BARRA_Q2 + 6)

    c_any = alt.Chart(pd.DataFrame({'t': [p]})).mark_text(
        fontSize=17, fontWeight='bold', align='center', color='#111111'
    ).encode(text='t:N').properties(width=AMPLADA_LABEL_Q2, height=ALCADA_BARRA_Q2 + 6)

    tot_p = int((df_q2['Trienni'] == p).sum())
    df_tot_p = pd.DataFrame({'x': [0], 't': [tot_p]})
    c_tot = (
        alt.Chart(df_tot_p).mark_circle(size=2200, color='#4a4a4a', opacity=0.92).encode(
            x=alt.X('x:Q', scale=alt.Scale(domain=[-1, 1]), axis=None)
        ) + alt.Chart(df_tot_p).mark_text(color='white', fontSize=12, fontWeight='bold').encode(
            x=alt.X('x:Q', scale=alt.Scale(domain=[-1, 1]), axis=None),
            text=alt.Text('t:Q', format=',')
        )
    ).properties(width=AMPLADA_BOLA_TOTAL_Q2, height=ALCADA_BARRA_Q2 + 6)

    # Referència explícita de les 3 files ocupant tota la vertical de la barra
    c_ref_files = alt.Chart(df_etiquetes_files_q2).mark_text(
        align='right', fontSize=11, fontWeight='bold', color='#444444'
    ).encode(
        y=alt.Y('y_row:Q', scale=alt.Scale(domain=[-0.5, 2.5], nice=False), axis=None),
        text='label_fila:N'
    ).properties(width=AMPLADA_REF_TRAM_Q2, height=ALCADA_BARRA_Q2 + 6)

    # Quadradets amb separació mínima en 3 files que ocupen tota la vertical de la barra
    sub_sq = df_sq_q2[df_sq_q2['Periode'] == p]
    guies_files = alt.Chart(df_etiquetes_files_q2).mark_rule(
        color='#eeeeee', strokeDash=[2, 2]
    ).encode(
        y=alt.Y('y_row:Q', scale=alt.Scale(domain=[-0.5, 2.5], nice=False), axis=None)
    )
    punts_sq = alt.Chart(sub_sq).mark_square(opacity=0.95, stroke='white', strokeWidth=0.3).encode(
        x=alt.X('col:Q', scale=alt.Scale(domain=[-0.8, max_col_q2], nice=False), axis=None),
        y=alt.Y('y_row:Q', scale=alt.Scale(domain=[-0.5, 2.5], nice=False), axis=None),
        size=alt.Size('mida_sq:Q', scale=alt.Scale(type='identity'), legend=None),
        color=alt.Color('Tram:N', scale=tram_q2_colors, legend=None),
        tooltip=[
            alt.Tooltip('Tram:N', title='Fila / Mida'),
            alt.Tooltip('Representa:N', title='Equivalència'),
            alt.Tooltip('Detall:N', title='Detall')
        ]
    )
    c_sq = (guies_files + punts_sq).properties(width=AMPLADA_PICTO_Q2, height=ALCADA_BARRA_Q2 + 6)

    files_q2.append(
        alt.hconcat(c_barra, c_any, c_tot, c_ref_files, c_sq, spacing=6).resolve_scale(color='independent')
    )

q2_final = alt.vconcat(*files_q2, spacing=4).properties(
    title=titol_rt(
        "Q2 · Evolució per triennis de les causes d'incendi i desglossament en 3 files per severitat (1998-2022)",
        "A la dreta, cada fila mostra els incendis Grans (1 quadradet = 1 GIF), Mitjans (1 quadradet = 20 focs) i Petits (1 quadradet = 20 conats)"
    )
).configure_view(strokeWidth=0).configure_axis(grid=False)
q2_final.save('Q2_trienis_quadradets_vg.html')


# %% ===================================================================
# Q3 · 9 Mapes (1r amb escala absoluta pròpia + 8 amb diferència absoluta vs. trienni anterior)
# =======================================================================
df_q3_comarca = (
    df.groupby(['PeriodeQ3', 'Nom comarca'], as_index=False, observed=True)
    .size().rename(columns={'size': 'Nombre_Incendis', 'PeriodeQ3': 'Periode'})
)
totes_comarques = sorted(list(noms_geojson))
index_complet_q3 = pd.MultiIndex.from_product([labels_q3, totes_comarques], names=['Periode', 'Nom comarca'])
df_q3_comarca = (
    df_q3_comarca.set_index(['Periode', 'Nom comarca'])
    .reindex(index_complet_q3, fill_value=0).reset_index()
)

# Calculem la diferència absoluta respecte al TRIENNI ANTERIOR per a cada comarca
df_q3_comarca['Incendis_Anterior'] = df_q3_comarca.groupby('Nom comarca')['Nombre_Incendis'].shift(1)
df_q3_comarca['Dif_Trienni_Anterior'] = df_q3_comarca['Nombre_Incendis'] - df_q3_comarca['Incendis_Anterior']

max_abs_p0 = float(df_q3_comarca[df_q3_comarca['Periode'] == labels_q3[0]]['Nombre_Incendis'].max())
lim_dif_ant = float(np.nanpercentile(df_q3_comarca['Dif_Trienni_Anterior'].abs(), 96))

w3, h3 = mida_mapa(340)
mapes_q3 = []

for idx_m, p in enumerate(labels_q3):
    dades_p = df_q3_comarca[df_q3_comarca['Periode'] == p]
    if idx_m == 0:
        # 1r trienni (1998-2000): no té històric previ -> Nombre absolut d'incendis amb escala pròpia
        m = alt.Chart(comarques_geo).mark_geoshape(stroke='white', strokeWidth=0.6).transform_lookup(
            lookup='properties.NOMCOMAR',
            from_=alt.LookupData(dades_p, 'Nom comarca', ['Nombre_Incendis'])
        ).encode(
            color=alt.Color(
                'Nombre_Incendis:Q',
                scale=alt.Scale(scheme='oranges', domain=[0, max_abs_p0]),
                title="Incendis 1998-2000 (base)",
                legend=alt.Legend(orient='bottom', gradientLength=180, titleFontSize=11, labelFontSize=10)
            ),
            tooltip=[
                alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
                alt.Tooltip('Nombre_Incendis:Q', title="Incendis al trienni inicial (1998-2000)", format=',')
            ]
        ).project(type=PROJECCIO, reflectY=REFLECTEIX_Y).properties(
            width=w3, height=h3,
            title=alt.TitleParams(f"{p} (Valors absoluts base)", fontSize=14, fontWeight='bold', color='#b35806')
        )
    else:
        # Resta de triennis: diferència absoluta respecte al trienni immediatament anterior
        p_ant = labels_q3[idx_m - 1]
        mostra_leg_dif = (idx_m == len(labels_q3) - 1)
        m = alt.Chart(comarques_geo).mark_geoshape(stroke='white', strokeWidth=0.6).transform_lookup(
            lookup='properties.NOMCOMAR',
            from_=alt.LookupData(
                dades_p, 'Nom comarca',
                ['Nombre_Incendis', 'Incendis_Anterior', 'Dif_Trienni_Anterior']
            )
        ).encode(
            color=alt.Color(
                'Dif_Trienni_Anterior:Q',
                scale=alt.Scale(
                    domain=[-lim_dif_ant, 0, lim_dif_ant],
                    range=['#542788', '#f7f7f7', '#e66101'],
                    clamp=True
                ),
                title="Variació vs. trienni anterior",
                legend=alt.Legend(orient='bottom', gradientLength=200, titleFontSize=11, labelFontSize=10)
                if mostra_leg_dif else None
            ),
            tooltip=[
                alt.Tooltip('properties.NOMCOMAR:N', title='Comarca'),
                alt.Tooltip('Nombre_Incendis:Q', title=f"Incendis ({p})", format=','),
                alt.Tooltip('Incendis_Anterior:Q', title=f"Incendis trienni previ ({p_ant})", format=','),
                alt.Tooltip('Dif_Trienni_Anterior:Q', title=f"Diferència ({p} vs {p_ant})", format='+,d')
            ]
        ).project(type=PROJECCIO, reflectY=REFLECTEIX_Y).properties(
            width=w3, height=h3,
            title=alt.TitleParams(f"{p} (vs. {p_ant})", fontSize=14, fontWeight='bold')
        )
    mapes_q3.append(m)

files_q3 = [
    alt.hconcat(*mapes_q3[i:i + 3], spacing=12).resolve_scale(color='independent')
    for i in range(0, 9, 3)
]
q3_final = alt.vconcat(*files_q3, spacing=12).resolve_scale(color='independent').properties(
    title=titol_rt(
        "Q3 · Evolució triennal per comarca: punt de partida (1998-2000) i variació absoluta respecte al trienni anterior",
        "El primer mapa mostra el volum absolut inicial (taronja); els 8 següents mostren l'augment (+ taronja) o descens (- púrpura) respecte al trienni previ"
    )
).configure_view(stroke=None)
q3_final.save('Q3_9mapes_dif_anterior_trieni_vg.html')


# %% ===================================================================
# Q4 · Esquerra: % ha per mida | ANY | Forestal vs No Forestal (barra) | Dreta: Boles (<500 ha vs >500 ha)
# =======================================================================
df_q4_mida_raw = (
    df.groupby(['PeriodeQ4', 'ClasseMida'], as_index=False, observed=True)
    .agg(Superficie=('SuperficieTotal', 'sum'), NumFocs=('SuperficieTotal', 'count'))
    .rename(columns={'PeriodeQ4': 'Periode'})
)

parts_mida_q4 = []
for idx_p, p in enumerate(labels_q4):
    sub = df_q4_mida_raw[df_q4_mida_raw['Periode'] == p].set_index('ClasseMida').reindex(classes_mida).reset_index()
    sub['Superficie'] = sub['Superficie'].fillna(0)
    sub['NumFocs'] = sub['NumFocs'].fillna(0)
    tot = sub['Superficie'].sum()
    sub['pct'] = sub['Superficie'] / tot if tot else 0
    sub['x1'] = sub['pct'].cumsum()
    sub['x0'] = sub['x1'] - sub['pct']
    sub['x_mig'] = (sub['x0'] + sub['x1']) / 2
    sub['Periode'] = p
    sub['label_pct'] = sub['pct'].apply(
        lambda v: (f"{round(v * 100)}%" if idx_p == 0 else f"{round(v * 100)}") if v >= 0.05 else ""
    )
    parts_mida_q4.append(sub)
df_q4_mida = pd.concat(parts_mida_q4, ignore_index=True)

df_q4_tipus_raw = (
    df.groupby('PeriodeQ4', as_index=False, observed=True)
    .agg(Forestal=('SuperficieTotalForestal', 'sum'), NoForestal=('SuperficieTotalNoForestal', 'sum'))
    .rename(columns={'PeriodeQ4': 'Periode'})
    .melt(id_vars='Periode', value_vars=['Forestal', 'NoForestal'], var_name='Tipus', value_name='Superficie')
)
tipus_domain = ['Forestal', 'NoForestal']
tipus_scale = alt.Scale(domain=tipus_domain, range=['#1b7837', '#b35806'])

parts_tipus_q4 = []
for idx_p, p in enumerate(labels_q4):
    sub = df_q4_tipus_raw[df_q4_tipus_raw['Periode'] == p].set_index('Tipus').reindex(tipus_domain).reset_index()
    sub['Superficie'] = sub['Superficie'].fillna(0)
    tot = sub['Superficie'].sum()
    sub['pct'] = sub['Superficie'] / tot if tot else 0
    sub['x1'] = sub['pct'].cumsum()
    sub['x0'] = sub['x1'] - sub['pct']
    sub['x_mig'] = (sub['x0'] + sub['x1']) / 2
    sub['Periode'] = p
    sub['label_pct'] = sub['pct'].apply(
        lambda v: (f"{round(v * 100)}%" if idx_p == 0 else f"{round(v * 100)}") if v >= 0.06 else ""
    )
    parts_tipus_q4.append(sub)
df_q4_tipus = pd.concat(parts_tipus_q4, ignore_index=True)

# --- Preparació de les dues columnes de boles a la dreta ---
# Columna 1 de boles: Resta d'incendis (<500 ha), estil q4cv2 (1 bola = 20 incendis, mida igual, color = classe)
GROUP_SIZE_Q4 = 20
COLS_RESTA_Q4 = 22
classes_resta = ['<1 ha', '1-10 ha', '10-100 ha', '100-500 ha']


def boles_resta_periode(subdf, periode):
    files = []
    idx_global = 0
    # Ordenem de més greu (100-500 ha) a menys greu (<1 ha) perquè els focs mitjans-grans es vegin primers
    for classe in reversed(classes_resta):
        n = int((subdf['ClasseMida'] == classe).sum())
        if n > 0 and classe == '100-500 ha':
            n_boles = max(1, int(round(n / GROUP_SIZE_Q4)))
        else:
            n_boles = int(n // GROUP_SIZE_Q4)
        for _ in range(n_boles):
            files.append({
                'Periode': periode,
                'ClasseMida': classe,
                'col': idx_global % COLS_RESTA_Q4,
                'fila': idx_global // COLS_RESTA_Q4,
                'Incendis_Reals': n,
                'Equivalencia': f"1 bola = {GROUP_SIZE_Q4} incendis ({classe})"
            })
            idx_global += 1
    return pd.DataFrame(files)


df_boles_resta = pd.concat(
    [boles_resta_periode(df[df['PeriodeQ4'] == p], p) for p in labels_q4],
    ignore_index=True
)
max_fila_resta = max(int(df_boles_resta['fila'].max()), 6)

# Columna 2 de boles: Incendis de >500 ha (INDIVIDUALS: 1 bola = 1 incendi)
COLS_GIF_Q4 = 5


def boles_gif_individuals(subdf, periode):
    files = []
    grans = subdf[subdf['SuperficieTotal'] >= 500].sort_values('SuperficieTotal', ascending=False)
    for i, (_, r) in enumerate(grans.iterrows()):
        mun = r.get('Nom municipi', r.get('Nom comarca', ''))
        if pd.isna(mun) or str(mun).strip() == '':
            mun = r.get('Nom comarca', '')
        files.append({
            'Periode': periode,
            'ClasseMida': '>500 ha',
            'col': i % COLS_GIF_Q4,
            'fila': i // COLS_GIF_Q4,
            'Nom_Comarca': r['Nom comarca'],
            'Ubicacio': f"{mun} ({int(r['Any'])})",
            'Any': int(r['Any']),
            'ha_real': float(r['SuperficieTotal'])
        })
    return pd.DataFrame(files)


df_boles_gif = pd.concat(
    [boles_gif_individuals(df[df['PeriodeQ4'] == p], p) for p in labels_q4],
    ignore_index=True
)
max_fila_gif = max(int(df_boles_gif['fila'].max()) if len(df_boles_gif) else 2, 3)

# Amplades de les 5 columnes de Q4 perquè càpiguen perfectament en pantalla de portàtil
ALCADA_FILA_Q4 = 68
W_Q4_MIDA = 360
W_Q4_ANY = 90
W_Q4_FORESTAL = 185
W_Q4_BOLES_RESTA = 320
W_Q4_BOLES_GIF = 145

# Fila de capçaleres Q4
cap_q4_mida = alt.Chart(pd.DataFrame({'ClasseMida': classes_mida})).mark_circle(size=160, opacity=0).encode(
    color=alt.Color(
        'ClasseMida:N', scale=mida_scale, title="% ha cremades per mida:",
        legend=alt.Legend(
            orient='top', direction='horizontal', columns=3,
            titleOrient='left', titleFontSize=12, labelFontSize=11,
            symbolType='circle', symbolSize=160, symbolOpacity=1
        )
    )
).properties(width=W_Q4_MIDA, height=1)

cap_q4_any = alt.Chart(pd.DataFrame({'t': ['PERÍODE']})).mark_text(
    fontSize=13, fontWeight='bold', color='#222222'
).encode(text='t:N').properties(width=W_Q4_ANY, height=1)

cap_q4_tipus = alt.Chart(pd.DataFrame({'Tipus': tipus_domain})).mark_circle(size=160, opacity=0).encode(
    color=alt.Color(
        'Tipus:N', scale=tipus_scale, title="Superfície (%):",
        legend=alt.Legend(
            orient='top', direction='horizontal',
            titleOrient='top', titleFontSize=12, labelFontSize=11,
            symbolType='circle', symbolSize=160, symbolOpacity=1
        )
    )
).properties(width=W_Q4_FORESTAL, height=1)

cap_q4_resta = alt.Chart(pd.DataFrame({'t': [f"Incendis <500 ha (1 bola = {GROUP_SIZE_Q4} focs)"]})).mark_text(
    fontSize=12, fontWeight='bold', align='left', color='#222222'
).encode(text='t:N').properties(width=W_Q4_BOLES_RESTA, height=1)

cap_q4_gif = alt.Chart(pd.DataFrame({'t': [">500 ha (1 bola = 1 GIF)"]})).mark_text(
    fontSize=12, fontWeight='bold', align='left', color='#1a1a1a'
).encode(text='t:N').properties(width=W_Q4_BOLES_GIF, height=1)

files_q4 = [alt.hconcat(cap_q4_mida, cap_q4_any, cap_q4_tipus, cap_q4_resta, cap_q4_gif, spacing=10)]

for p in labels_q4:
    # 1. Esquerra: Stacked bars de % d'hectàrees per mida
    sub_m = df_q4_mida[df_q4_mida['Periode'] == p]
    c_mida = (
        alt.Chart(sub_m).mark_bar(height=ALCADA_FILA_Q4).encode(
            x=alt.X('x0:Q', scale=alt.Scale(domain=[0, 1], nice=False), axis=None),
            x2='x1:Q',
            color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None),
            tooltip=[
                alt.Tooltip('Periode:N', title='Període'),
                alt.Tooltip('ClasseMida:N', title='Mida incendi'),
                alt.Tooltip('Superficie:Q', title='Hectàrees cremades (ha)', format=',.1f'),
                alt.Tooltip('pct:Q', title='% de superfície', format='.1%')
            ]
        ) + alt.Chart(sub_m[sub_m['pct'] >= 0.05]).mark_text(
            color='white', fontSize=13, fontWeight='bold', baseline='middle'
        ).encode(
            x=alt.X('x_mig:Q', scale=alt.Scale(domain=[0, 1], nice=False)),
            text='label_pct:N'
        )
    ).properties(width=W_Q4_MIDA, height=ALCADA_FILA_Q4 + 8)

    # 2. Any
    c_any = alt.Chart(pd.DataFrame({'t': [p]})).mark_text(
        fontSize=17, fontWeight='bold', align='center', color='#111111'
    ).encode(text='t:N').properties(width=W_Q4_ANY, height=ALCADA_FILA_Q4 + 8)

    # 3. Forestal vs No Forestal (només bar chart, sense pie chart)
    sub_t = df_q4_tipus[df_q4_tipus['Periode'] == p]
    c_forestal = (
        alt.Chart(sub_t).mark_bar(height=ALCADA_FILA_Q4).encode(
            x=alt.X('x0:Q', scale=alt.Scale(domain=[0, 1], nice=False), axis=None),
            x2='x1:Q',
            color=alt.Color('Tipus:N', scale=tipus_scale, legend=None),
            tooltip=[
                alt.Tooltip('Periode:N', title='Període'),
                alt.Tooltip('Tipus:N', title='Cobertura'),
                alt.Tooltip('Superficie:Q', title='Hectàrees (ha)', format=',.1f'),
                alt.Tooltip('pct:Q', title='Percentatge', format='.1%')
            ]
        ) + alt.Chart(sub_t[sub_t['pct'] >= 0.06]).mark_text(
            color='white', fontSize=12, fontWeight='bold', baseline='middle'
        ).encode(
            x=alt.X('x_mig:Q', scale=alt.Scale(domain=[0, 1], nice=False)),
            text='label_pct:N'
        )
    ).properties(width=W_Q4_FORESTAL, height=ALCADA_FILA_Q4 + 8)

    # 4. Columna 1 de boles: Tota la resta (<500 ha), 1 bola = 20 incendis
    sub_r = df_boles_resta[df_boles_resta['Periode'] == p]
    c_boles_resta = alt.Chart(sub_r).mark_circle(size=55, opacity=0.95, stroke='white', strokeWidth=0.4).encode(
        x=alt.X('col:Q', scale=alt.Scale(domain=[-0.8, COLS_RESTA_Q4], nice=False), axis=None),
        y=alt.Y('fila:Q', scale=alt.Scale(domain=[-0.8, max_fila_resta + 0.8], reverse=True, nice=False), axis=None),
        color=alt.Color('ClasseMida:N', scale=mida_scale, legend=None),
        tooltip=[
            alt.Tooltip('Periode:N', title='Període'),
            alt.Tooltip('ClasseMida:N', title='Categoria'),
            alt.Tooltip('Incendis_Reals:Q', title="Total incendis d'aquesta classe", format=','),
            alt.Tooltip('Equivalencia:N', title='Escala')
        ]
    ).properties(width=W_Q4_BOLES_RESTA, height=ALCADA_FILA_Q4 + 8)

    # 5. Columna 2 de boles: Incendis >500 ha (individuals: 1 bola = 1 incendi)
    sub_g = df_boles_gif[df_boles_gif['Periode'] == p]
    fons_gif = alt.Chart(pd.DataFrame({'x': [0]})).mark_rect(
        fill='#f8f8f8', stroke='#e0e0e0', strokeDash=[3, 3], cornerRadius=4
    ).properties(width=W_Q4_BOLES_GIF, height=ALCADA_FILA_Q4 + 8)

    punts_gif = alt.Chart(sub_g).mark_circle(size=105, color='#1a1a1a', opacity=0.95, stroke='#d73027', strokeWidth=1.2).encode(
        x=alt.X('col:Q', scale=alt.Scale(domain=[-0.7, COLS_GIF_Q4 - 0.3], nice=False), axis=None),
        y=alt.Y('fila:Q', scale=alt.Scale(domain=[-0.7, max_fila_gif + 0.5], reverse=True, nice=False), axis=None),
        tooltip=[
            alt.Tooltip('Ubicacio:N', title='Gran Incendi Forestal (GIF)'),
            alt.Tooltip('Nom_Comarca:N', title='Comarca'),
            alt.Tooltip('Any:Q', title='Any'),
            alt.Tooltip('ha_real:Q', title='Hectàrees cremades (ha)', format=',.1f')
        ]
    ).properties(width=W_Q4_BOLES_GIF, height=ALCADA_FILA_Q4 + 8)

    c_boles_gif = fons_gif + punts_gif

    files_q4.append(
        alt.hconcat(c_mida, c_any, c_forestal, c_boles_resta, c_boles_gif, spacing=10).resolve_scale(color='independent')
    )

q4_final = alt.vconcat(*files_q4, spacing=4).properties(
    title=titol_rt(
        "Q4 · Un grapat d'incendis individuals (>500 ha) dictaminen el balanç d'hectàrees cremades de cada període",
        "D'esquerra a dreta: % d'hectàrees per mida, període, % forestal vs. no forestal, boles agrupades (<500 ha, 1=20) i boles individuals (>500 ha, 1=1 GIF)"
    )
).configure_view(strokeWidth=0).configure_axis(grid=False)
q4_final.save('Q4_combo_final_vg.html')


# %% ===================================================================
# RESUM DE FITXERS GENERATS
# =======================================================================
fitxers_finals = [
    'Q1_v1_dos_mapes_vg.html',
    'Q1_v2_distorsio_cartograma_vg.html',
    'Q2_trienis_quadradets_vg.html',
    'Q3_9mapes_dif_anterior_trieni_vg.html',
    'Q4_combo_final_vg.html'
]

print("\nFitxers finals generats correctament:")
for nom in fitxers_finals:
    print(f"  - {nom}")